# 04. Model Evaluation

This notebook loads the saved model, recreates the same test split, calculates regression metrics, and compares actual and predicted delivery times.

First run `03_model_training.ipynb`.

## 1. Import libraries

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load and prepare the dataset

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../dataset/foodtimedataset.csv")

df.columns = df.columns.str.strip()

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

df["Delivery Time"] = pd.to_numeric(df["Delivery Time"], errors="coerce")
df = df.dropna(subset=["Delivery Time"])
df = df[df["Delivery Time"] > 0]

print("Cleaned dataset shape:", df.shape)
df.head()

Cleaned dataset shape: (9040, 17)


,ID,Delivery Person ID,Delivery Person Age,Delivery Person Ratings,Restaurant Latitude,Restaurant Longitude,Delivery Location Latitude,Delivery Location Longitude,Order Type,Vehicle Type,Temperature,Humidity,Precipitation,Weather Description,Traffic Level,Distance,Delivery Time
0,4607,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,Snack,motorcycle,17.11,77.0,0.0,haze,Low,NaN,21.666667
1,B379,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,Snack,scooter,19.50,93.0,0.0,mist,Very High,37.17,85.266667
2,5D6D,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,Drinks,motorcycle,20.45,91.0,0.0,mist,Low,3.34,28.583333
3,7A6A,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,Buffet,motorcycle,23.86,78.0,0.0,mist,Moderate,10.05,35.183333
4,70A2,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,Snack,scooter,26.55,87.0,0.0,mist,High,9.89,43.450000


In [3]:
X = df.drop(columns=["ID", "Delivery Person ID", "Delivery Time"])
y = df["Delivery Time"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

## 3. Load model and predict

In [4]:
model = joblib.load("../pickle_file/food_delivery_xgboost.pkl")
predictions = model.predict(X_test)

print("Predictions generated successfully!")

Predictions generated successfully!


## 4. Calculate model performance

In [5]:
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)
mse = mean_squared_error(y_test, predictions)

print("XGBoost Model Results")
print("MAE:", round(mae, 2), "minutes")
print("RMSE:", round(rmse, 2), "minutes")
print("MSE:", round(mse, 2))
print("R2 Score:", round(r2, 4))
print("R2 Score:", round(r2, 4))

XGBoost Model Results
MAE: 2.23 minutes
RMSE: 3.04 minutes
MSE: 9.26
R2 Score: 0.9656
R2 Score: 0.9656


## 5. Compare actual and predicted delivery times

In [6]:
results = pd.DataFrame({
    "Actual Delivery Time": y_test.iloc[:10].values,
    "Predicted Delivery Time": predictions[:10].round(2)
})

results

,Actual Delivery Time,Predicted Delivery Time
0,36.966667,34.400002
1,63.750000,62.750000
2,39.050000,42.619999
3,36.133333,36.009998
4,34.000000,35.369999
5,23.500000,22.320000
6,27.100000,26.400000
7,30.100000,28.530001
8,48.300000,44.619999
9,24.200000,25.570000
